<a href="https://colab.research.google.com/github/chenyuan9977/med-ner-capstone/blob/main/medical_ner_main_chenyuan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Medical NER with Contrastive Learning
## CS687 Capstone Project - Chen Yuan

**Two-Stage Architecture:**
1. **Stage 1**: BioBERT NER Tagger with BIO tagging
2. **Stage 2**: Contrastive Learning for Entity Normalization + FAISS Retrieval

**Dataset**: NCBI Disease Corpus (793 PubMed abstracts, ~6,900 disease mentions)



#  SECTION 1: SETUP AND INSTALLATION


In [ ]:
#
# Install Dependencies
#
import os

print("Installing dependencies...")
os.system('pip install -q -U transformers accelerate datasets==2.16.0 seqeval')
os.system('pip install -q sentence-transformers faiss-cpu')
os.system('pip install -q matplotlib seaborn scikit-learn')

import torch
import json
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import defaultdict
import time
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from tqdm.auto import tqdm
import faiss

print("\n✅ All libraries installed and imported successfully!")

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

Installing dependencies...


In [ ]:
#
# Configuration and Mount Drive


class Config:
    # Paths
    MOUNT_PATH = '/content/drive'
    PROJECT_DIR = '/content/drive/MyDrive/medical_ner'

    # Stage 1: NER Model
    NER_MODEL = "dmis-lab/biobert-v1.1"
    MAX_LEN = 128
    BATCH_SIZE = 16
    EPOCHS_NER = 5
    LR_NER = 2e-5

    # Stage 2: Contrastive Learning
    CONTRASTIVE_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
    EPOCHS_CONTRASTIVE = 5
    LR_CONTRASTIVE = 1e-5
    TEMPERATURE = 0.05
    BATCH_SIZE_CONTRASTIVE = 32

    # Random seed
    SEED = 42

# Set seeds
torch.manual_seed(Config.SEED)
np.random.seed(Config.SEED)
random.seed(Config.SEED)

# Mount Google Drive
if not os.path.exists(Config.MOUNT_PATH):
    drive.mount(Config.MOUNT_PATH)

# Create directories
os.makedirs(Config.PROJECT_DIR, exist_ok=True)
os.makedirs(f"{Config.PROJECT_DIR}/models", exist_ok=True)
os.makedirs(f"{Config.PROJECT_DIR}/results", exist_ok=True)

print(f"📁 Project directory: {Config.PROJECT_DIR}")

Mounted at /content/drive
📁 Project directory: /content/drive/MyDrive/medical_ner


#  SECTION 2: LOAD NCBI DISEASE DATASET

In [ ]:
#
#  Load NCBI Disease Dataset
#

print("📥 Loading NCBI Disease dataset...")

dataset = load_dataset("ncbi_disease", trust_remote_code=True)

print(f"\n✅ Dataset loaded!")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Validation: {len(dataset['validation'])} samples")
print(f"   Test: {len(dataset['test'])} samples")

📥 Loading NCBI Disease dataset...


Generating train split:   0%|          | 0/5433 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/924 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/941 [00:00<?, ? examples/s]


✅ Dataset loaded!
   Train: 5433 samples
   Validation: 924 samples
   Test: 941 samples


In [ ]:
#
# CELL 2.2: Explore the Dataset
#

# Get label names
label_names = dataset['train'].features['ner_tags'].feature.names
print(f"🏷️ NER Labels: {label_names}")

# Create label mappings
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

print(f"\n📊 Label mapping: {label2id}")

# Show example
print("\n📝 Example from training set:")
example = dataset['train'][0]
print(f"   Tokens: {example['tokens'][:10]}...")
print(f"   Tags: {[label_names[t] for t in example['ner_tags'][:10]]}...")

🏷️ NER Labels: ['O', 'B-Disease', 'I-Disease']

📊 Label mapping: {'O': 0, 'B-Disease': 1, 'I-Disease': 2}

📝 Example from training set:
   Tokens: ['Identification', 'of', 'APC2', ',', 'a', 'homologue', 'of', 'the', 'adenomatous', 'polyposis']...
   Tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Disease', 'I-Disease']...


# SECTION 3: STAGE 1 - NER WITH BIOBERT


In [ ]:
#
# Tokenize Dataset for NER
#

print(f"🔄 Loading tokenizer: {Config.NER_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(Config.NER_MODEL)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=Config.MAX_LEN,
        padding='max_length'
    )

    all_labels = []
    for i, labels in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label = labels[word_idx]
                if label == 1:  # B-Disease -> I-Disease
                    label_ids.append(2)
                else:
                    label_ids.append(label)
            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized['labels'] = all_labels
    return tokenized

print("⏳ Tokenizing datasets...")
tokenized_train = dataset['train'].map(tokenize_and_align_labels, batched=True)
tokenized_val = dataset['validation'].map(tokenize_and_align_labels, batched=True)
tokenized_test = dataset['test'].map(tokenize_and_align_labels, batched=True)
print("✅ Tokenization complete!")

🔄 Loading tokenizer: dmis-lab/biobert-v1.1


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

⏳ Tokenizing datasets...


Map:   0%|          | 0/5433 [00:00<?, ? examples/s]

Map:   0%|          | 0/924 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

✅ Tokenization complete!


In [ ]:
#
# Define Evaluation Metrics
#
from seqeval.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    pred_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        true_seq = []
        pred_seq_clean = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                true_seq.append(id2label[l])
                pred_seq_clean.append(id2label[p])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq_clean)

    return {
        'precision': precision_score(true_labels, pred_labels),
        'recall': recall_score(true_labels, pred_labels),
        'f1': f1_score(true_labels, pred_labels)
    }

print("✅ Evaluation metrics defined!")

✅ Evaluation metrics defined!


In [ ]:
#
# Train NER Model
#

print(f"🔄 Loading BioBERT model: {Config.NER_MODEL}")

model = AutoModelForTokenClassification.from_pretrained(
    Config.NER_MODEL,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=f"{Config.PROJECT_DIR}/ner_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=Config.LR_NER,
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    num_train_epochs=Config.EPOCHS_NER,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("\n🚀 Starting Stage 1 Training (NER)...")
trainer.train()

# Save model
trainer.save_model(f"{Config.PROJECT_DIR}/models/stage1_ner")
tokenizer.save_pretrained(f"{Config.PROJECT_DIR}/models/stage1_ner")
print("\n✅ Stage 1 NER Model trained and saved!")

🔄 Loading BioBERT model: dmis-lab/biobert-v1.1


pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Stage 1 NER Model trained and saved!


In [ ]:
#
#  Evaluate Stage 1 on Test Set
#

from transformers.utils.notebook import NotebookProgressCallback

print("📊 Evaluating Stage 1 NER on Test Set...")

trainer.remove_callback(NotebookProgressCallback)
test_results = trainer.evaluate(tokenized_test)

print("\n" + "=" * 50)
print("📊 STAGE 1 TEST RESULTS")
print("=" * 50)
print(f"   Precision: {test_results['eval_precision']:.4f}")
print(f"   Recall:    {test_results['eval_recall']:.4f}")
print(f"   F1 Score:  {test_results['eval_f1']:.4f}")
print("=" * 50)

stage1_results = {
    'precision': test_results['eval_precision'],
    'recall': test_results['eval_recall'],
    'f1': test_results['eval_f1']
}

📊 Evaluating Stage 1 NER on Test Set...

📊 STAGE 1 TEST RESULTS
   Precision: 0.8330
   Recall:    0.8990
   F1 Score:  0.8647


In [ ]:
#
# Test NER Predictions
#
from transformers import pipeline

print("🔬 Testing NER predictions...")

ner_pipeline = pipeline(
    "ner",
    model=f"{Config.PROJECT_DIR}/models/stage1_ner",
    tokenizer=f"{Config.PROJECT_DIR}/models/stage1_ner",
    aggregation_strategy="simple"
)

test_sentences = [
    "The patient was diagnosed with diabetes mellitus and hypertension.",
    "Parkinson's disease affects the central nervous system.",
    "Treatment for lung cancer includes chemotherapy.",
]

print("\n" + "="*60)
for sentence in test_sentences:
    print(f"\n📝 Input: {sentence}")
    results = ner_pipeline(sentence)
    if results:
        for ent in results:
            print(f"   🏷️ '{ent['word']}' ({ent['entity_group']}) [score: {ent['score']:.3f}]")
    else:
        print("   No entities detected.")
print("="*60)

🔬 Testing NER predictions...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]



📝 Input: The patient was diagnosed with diabetes mellitus and hypertension.
   🏷️ 'diabetes mellitus' (Disease) [score: 0.999]
   🏷️ 'hypertension' (Disease) [score: 0.998]

📝 Input: Parkinson's disease affects the central nervous system.
   🏷️ 'Parkinson ' s disease' (Disease) [score: 0.999]

📝 Input: Treatment for lung cancer includes chemotherapy.
   🏷️ 'lung cancer' (Disease) [score: 0.990]


#  SECTION 4: STAGE 2 - CONTRASTIVE LEARNING


In [ ]:
#
#  Extract Disease Entities for Candidate Bank
#

def extract_entities_from_dataset(dataset_split):
    """Extract all disease entities from the dataset."""
    entities = set()

    for sample in dataset_split:
        tokens = sample['tokens']
        tags = sample['ner_tags']

        current_entity = []
        for token, tag in zip(tokens, tags):
            if tag == 1:  # B-Disease
                if current_entity:
                    entities.add(' '.join(current_entity))
                current_entity = [token]
            elif tag == 2 and current_entity:  # I-Disease
                current_entity.append(token)
            else:
                if current_entity:
                    entities.add(' '.join(current_entity))
                    current_entity = []
        if current_entity:
            entities.add(' '.join(current_entity))

    return list(entities)

print(" Extracting disease entities...")
train_entities = extract_entities_from_dataset(dataset['train'])
val_entities = extract_entities_from_dataset(dataset['validation'])
test_entities = extract_entities_from_dataset(dataset['test'])

# Create candidate bank
candidate_bank = list(set(train_entities + val_entities))

print(f"✅ Candidate Bank: {len(candidate_bank)} unique entities")
print(f"   Sample: {candidate_bank[:5]}")

 Extracting disease entities...
✅ Candidate Bank: 1902 unique entities
   Sample: ['complex glycerol kinase deficiency', 'galactosemia', 'Autosomal dominant neurohypophyseal diabetes insipidus', 'neuromuscular diseases', 'Van der Woude syndrome']


In [ ]:
#
# Comprehensive Medical Synonym Training Pairs
#

training_synonyms = [

    # ===========================================
    # CARDIOVASCULAR DISEASES
    # ===========================================
    ("heart attack", "myocardial infarction"),
    ("heart attack", "MI"),
    ("heart attack", "cardiac infarction"),
    ("high blood pressure", "hypertension"),
    ("high blood pressure", "HTN"),
    ("elevated blood pressure", "hypertension"),
    ("low blood pressure", "hypotension"),
    ("chest pain", "angina pectoris"),
    ("chest pain", "angina"),
    ("irregular heartbeat", "arrhythmia"),
    ("irregular heartbeat", "cardiac arrhythmia"),
    ("fast heartbeat", "tachycardia"),
    ("slow heartbeat", "bradycardia"),
    ("heart failure", "cardiac failure"),
    ("heart failure", "congestive heart failure"),
    ("heart failure", "CHF"),
    ("hardening of arteries", "atherosclerosis"),
    ("clogged arteries", "atherosclerosis"),
    ("enlarged heart", "cardiomegaly"),
    ("heart inflammation", "myocarditis"),
    ("heart muscle disease", "cardiomyopathy"),
    ("blood clot", "thrombosis"),
    ("blood clot in leg", "deep vein thrombosis"),
    ("blood clot in leg", "DVT"),
    ("blood clot in lung", "pulmonary embolism"),
    ("blood clot in lung", "PE"),

    # ===========================================
    # NEUROLOGICAL DISEASES
    # ===========================================
    ("stroke", "cerebrovascular accident"),
    ("stroke", "CVA"),
    ("stroke", "brain attack"),
    ("mini stroke", "transient ischemic attack"),
    ("mini stroke", "TIA"),
    ("seizure", "epileptic seizure"),
    ("seizure", "convulsion"),
    ("seizure disorder", "epilepsy"),
    ("memory loss", "amnesia"),
    ("confusion", "delirium"),
    ("paralysis", "palsy"),
    ("paralysis on one side", "hemiplegia"),
    ("weakness on one side", "hemiparesis"),
    ("shaking", "tremor"),
    ("headache", "cephalalgia"),
    ("migraine headache", "migraine"),
    ("dizziness", "vertigo"),
    ("fainting", "syncope"),
    ("numbness", "paresthesia"),
    ("tingling", "paresthesia"),
    ("nerve pain", "neuralgia"),
    ("brain inflammation", "encephalitis"),
    ("membrane inflammation", "meningitis"),
    ("dementia", "cognitive decline"),
    ("alzheimer's", "alzheimer's disease"),
    ("alzheimer's", "alzheimer disease"),
    ("alzheimer's", "AD"),
    ("parkinson's", "parkinson's disease"),
    ("parkinson's", "parkinson disease"),
    ("parkinson's", "PD"),
    ("lou gehrig's disease", "amyotrophic lateral sclerosis"),
    ("lou gehrig's disease", "ALS"),
    ("multiple sclerosis", "MS"),
    ("brain tumor", "cerebral neoplasm"),

    # ===========================================
    # RESPIRATORY DISEASES
    # ===========================================
    ("flu", "influenza"),
    ("common cold", "upper respiratory infection"),
    ("common cold", "URI"),
    ("whooping cough", "pertussis"),
    ("lung infection", "pneumonia"),
    ("bronchitis", "bronchial inflammation"),
    ("asthma", "bronchial asthma"),
    ("asthma attack", "asthma exacerbation"),
    ("copd", "chronic obstructive pulmonary disease"),
    ("emphysema", "pulmonary emphysema"),
    ("collapsed lung", "pneumothorax"),
    ("fluid in lungs", "pulmonary edema"),
    ("lung scarring", "pulmonary fibrosis"),
    ("tuberculosis", "TB"),
    ("shortness of breath", "dyspnea"),
    ("difficulty breathing", "dyspnea"),
    ("rapid breathing", "tachypnea"),
    ("coughing up blood", "hemoptysis"),
    ("sleep apnea", "obstructive sleep apnea"),
    ("snoring disorder", "sleep apnea"),
    ("lung cancer", "pulmonary carcinoma"),
    ("lung cancer", "bronchogenic carcinoma"),
    ("chest infection", "respiratory infection"),
    ("sinus infection", "sinusitis"),
    ("throat infection", "pharyngitis"),
    ("sore throat", "pharyngitis"),
    ("voice box inflammation", "laryngitis"),
    ("croup", "laryngotracheobronchitis"),

    # ===========================================
    # METABOLIC / ENDOCRINE DISEASES
    # ===========================================
    ("diabetes", "diabetes mellitus"),
    ("diabetes", "DM"),
    ("sugar disease", "diabetes mellitus"),
    ("type 1 diabetes", "insulin-dependent diabetes"),
    ("type 1 diabetes", "IDDM"),
    ("type 2 diabetes", "non-insulin-dependent diabetes"),
    ("type 2 diabetes", "NIDDM"),
    ("low blood sugar", "hypoglycemia"),
    ("high blood sugar", "hyperglycemia"),
    ("thyroid problem", "thyroid disorder"),
    ("overactive thyroid", "hyperthyroidism"),
    ("underactive thyroid", "hypothyroidism"),
    ("thyroid inflammation", "thyroiditis"),
    ("goiter", "thyroid enlargement"),
    ("high cholesterol", "hypercholesterolemia"),
    ("high cholesterol", "hyperlipidemia"),
    ("gout", "gouty arthritis"),
    ("gout", "hyperuricemia"),
    ("obesity", "morbid obesity"),
    ("overweight", "obesity"),
    ("metabolic syndrome", "syndrome X"),
    ("addison's disease", "adrenal insufficiency"),
    ("cushing's syndrome", "hypercortisolism"),

    # ===========================================
    # GASTROINTESTINAL DISEASES
    # ===========================================
    ("stomach flu", "gastroenteritis"),
    ("stomach bug", "gastroenteritis"),
    ("food poisoning", "foodborne illness"),
    ("heartburn", "acid reflux"),
    ("heartburn", "gastroesophageal reflux"),
    ("heartburn", "GERD"),
    ("stomach ulcer", "peptic ulcer"),
    ("stomach ulcer", "gastric ulcer"),
    ("liver disease", "hepatic disease"),
    ("liver inflammation", "hepatitis"),
    ("fatty liver", "hepatic steatosis"),
    ("liver scarring", "cirrhosis"),
    ("liver failure", "hepatic failure"),
    ("yellow skin", "jaundice"),
    ("yellow eyes", "jaundice"),
    ("gallstones", "cholelithiasis"),
    ("gallbladder inflammation", "cholecystitis"),
    ("pancreatitis", "pancreatic inflammation"),
    ("appendicitis", "appendix inflammation"),
    ("stomach inflammation", "gastritis"),
    ("intestinal inflammation", "enteritis"),
    ("colon inflammation", "colitis"),
    ("crohn's disease", "inflammatory bowel disease"),
    ("crohn's disease", "IBD"),
    ("ulcerative colitis", "inflammatory bowel disease"),
    ("irritable bowel", "irritable bowel syndrome"),
    ("irritable bowel", "IBS"),
    ("constipation", "obstipation"),
    ("diarrhea", "loose stools"),
    ("bloody stool", "hematochezia"),
    ("vomiting blood", "hematemesis"),
    ("difficulty swallowing", "dysphagia"),
    ("hemorrhoids", "piles"),
    ("colon cancer", "colorectal cancer"),
    ("colon cancer", "bowel cancer"),

    # ===========================================
    # KIDNEY / URINARY DISEASES
    # ===========================================
    ("kidney failure", "renal failure"),
    ("kidney disease", "renal disease"),
    ("chronic kidney disease", "CKD"),
    ("kidney inflammation", "nephritis"),
    ("kidney infection", "pyelonephritis"),
    ("kidney stones", "renal calculi"),
    ("kidney stones", "nephrolithiasis"),
    ("bladder infection", "cystitis"),
    ("urinary tract infection", "UTI"),
    ("painful urination", "dysuria"),
    ("frequent urination", "polyuria"),
    ("blood in urine", "hematuria"),
    ("protein in urine", "proteinuria"),
    ("bed wetting", "enuresis"),
    ("inability to urinate", "urinary retention"),
    ("incontinence", "urinary incontinence"),
    ("enlarged prostate", "benign prostatic hyperplasia"),
    ("enlarged prostate", "BPH"),
    ("prostate cancer", "prostatic carcinoma"),

    # ===========================================
    # CANCER
    # ===========================================
    ("cancer", "malignancy"),
    ("cancer", "carcinoma"),
    ("cancer", "neoplasm"),
    ("tumor", "neoplasm"),
    ("benign tumor", "benign neoplasm"),
    ("malignant tumor", "malignant neoplasm"),
    ("skin cancer", "melanoma"),
    ("skin cancer", "cutaneous carcinoma"),
    ("breast cancer", "breast carcinoma"),
    ("breast cancer", "mammary carcinoma"),
    ("lung cancer", "pulmonary carcinoma"),
    ("colon cancer", "colorectal carcinoma"),
    ("liver cancer", "hepatocellular carcinoma"),
    ("liver cancer", "HCC"),
    ("stomach cancer", "gastric carcinoma"),
    ("pancreatic cancer", "pancreatic carcinoma"),
    ("kidney cancer", "renal cell carcinoma"),
    ("bladder cancer", "urothelial carcinoma"),
    ("brain cancer", "brain tumor"),
    ("brain cancer", "glioma"),
    ("blood cancer", "leukemia"),
    ("lymph node cancer", "lymphoma"),
    ("bone marrow cancer", "multiple myeloma"),
    ("thyroid cancer", "thyroid carcinoma"),
    ("ovarian cancer", "ovarian carcinoma"),
    ("uterine cancer", "endometrial carcinoma"),
    ("cervical cancer", "cervical carcinoma"),
    ("testicular cancer", "testicular carcinoma"),
    ("bone cancer", "osteosarcoma"),
    ("cancer spread", "metastasis"),

    # ===========================================
    # INFECTIOUS DISEASES
    # ===========================================
    ("aids", "acquired immunodeficiency syndrome"),
    ("hiv", "human immunodeficiency virus"),
    ("german measles", "rubella"),
    ("measles", "rubeola"),
    ("chickenpox", "varicella"),
    ("shingles", "herpes zoster"),
    ("mono", "mononucleosis"),
    ("mono", "infectious mononucleosis"),
    ("kissing disease", "mononucleosis"),
    ("cold sore", "herpes simplex"),
    ("cold sore", "oral herpes"),
    ("staph infection", "staphylococcal infection"),
    ("strep throat", "streptococcal pharyngitis"),
    ("mrsa", "methicillin-resistant staphylococcus aureus"),
    ("tetanus", "lockjaw"),
    ("rabies", "hydrophobia"),
    ("malaria", "plasmodium infection"),
    ("lyme disease", "lyme borreliosis"),
    ("rocky mountain spotted fever", "RMSF"),
    ("west nile virus", "west nile fever"),
    ("dengue", "dengue fever"),
    ("yellow fever", "yellow jack"),
    ("ebola", "ebola virus disease"),
    ("covid", "coronavirus disease"),
    ("covid-19", "coronavirus disease 2019"),
    ("coronavirus", "SARS-CoV-2"),
    ("fungal infection", "mycosis"),
    ("yeast infection", "candidiasis"),
    ("ringworm", "tinea"),
    ("athlete's foot", "tinea pedis"),
    ("jock itch", "tinea cruris"),
    ("pinworms", "enterobiasis"),
    ("tapeworm", "taeniasis"),
    ("hookworm", "ancylostomiasis"),

    # ===========================================
    # MUSCULOSKELETAL DISEASES
    # ===========================================
    ("joint pain", "arthralgia"),
    ("muscle pain", "myalgia"),
    ("back pain", "dorsalgia"),
    ("neck pain", "cervicalgia"),
    ("bone loss", "osteoporosis"),
    ("soft bones", "osteomalacia"),
    ("brittle bones", "osteogenesis imperfecta"),
    ("joint inflammation", "arthritis"),
    ("rheumatoid arthritis", "RA"),
    ("osteoarthritis", "degenerative joint disease"),
    ("osteoarthritis", "OA"),
    ("wear and tear arthritis", "osteoarthritis"),
    ("lupus", "systemic lupus erythematosus"),
    ("lupus", "SLE"),
    ("fibromyalgia", "fibromyalgia syndrome"),
    ("muscle weakness", "myopathy"),
    ("muscle wasting", "muscular atrophy"),
    ("muscular dystrophy", "MD"),
    ("tendon inflammation", "tendinitis"),
    ("bursa inflammation", "bursitis"),
    ("carpal tunnel", "carpal tunnel syndrome"),
    ("slipped disc", "herniated disc"),
    ("slipped disc", "disc herniation"),
    ("sciatica", "sciatic nerve pain"),
    ("scoliosis", "spinal curvature"),
    ("spinal stenosis", "narrowing of spinal canal"),
    ("fracture", "broken bone"),
    ("sprain", "ligament injury"),
    ("strain", "muscle strain"),
    ("dislocation", "joint dislocation"),

    # ===========================================
    # SKIN DISEASES
    # ===========================================
    ("eczema", "atopic dermatitis"),
    ("psoriasis", "psoriatic disease"),
    ("acne", "acne vulgaris"),
    ("hives", "urticaria"),
    ("rash", "dermatitis"),
    ("skin inflammation", "dermatitis"),
    ("contact dermatitis", "allergic dermatitis"),
    ("diaper rash", "diaper dermatitis"),
    ("dandruff", "seborrheic dermatitis"),
    ("hair loss", "alopecia"),
    ("baldness", "alopecia"),
    ("vitiligo", "skin depigmentation"),
    ("warts", "verruca"),
    ("moles", "nevi"),
    ("birthmark", "nevus"),
    ("boil", "furuncle"),
    ("abscess", "skin abscess"),
    ("cellulitis", "skin infection"),
    ("impetigo", "skin infection"),
    ("shingles rash", "herpes zoster"),
    ("cold sores", "herpes labialis"),
    ("bedsore", "pressure ulcer"),
    ("bedsore", "decubitus ulcer"),
    ("stretch marks", "striae"),
    ("bruise", "contusion"),
    ("burn", "thermal injury"),
    ("sunburn", "solar erythema"),
    ("frostbite", "cold injury"),
    ("itching", "pruritus"),
    ("dry skin", "xerosis"),
    ("oily skin", "seborrhea"),
    ("skin cancer", "cutaneous malignancy"),

    # ===========================================
    # MENTAL HEALTH
    # ===========================================
    ("depression", "major depressive disorder"),
    ("depression", "MDD"),
    ("sadness disorder", "depression"),
    ("clinical depression", "major depression"),
    ("anxiety", "anxiety disorder"),
    ("worry disorder", "generalized anxiety disorder"),
    ("panic attacks", "panic disorder"),
    ("social anxiety", "social phobia"),
    ("fear of crowds", "agoraphobia"),
    ("fear of heights", "acrophobia"),
    ("fear of spiders", "arachnophobia"),
    ("ocd", "obsessive compulsive disorder"),
    ("ptsd", "post-traumatic stress disorder"),
    ("bipolar", "bipolar disorder"),
    ("mood swings", "bipolar disorder"),
    ("manic depression", "bipolar disorder"),
    ("schizophrenia", "psychotic disorder"),
    ("split personality", "dissociative identity disorder"),
    ("eating disorder", "anorexia nervosa"),
    ("binge eating", "bulimia nervosa"),
    ("adhd", "attention deficit hyperactivity disorder"),
    ("add", "attention deficit disorder"),
    ("autism", "autism spectrum disorder"),
    ("autism", "ASD"),
    ("learning disability", "learning disorder"),
    ("dyslexia", "reading disorder"),
    ("insomnia", "sleep disorder"),
    ("sleep problems", "insomnia"),
    ("addiction", "substance use disorder"),
    ("alcoholism", "alcohol use disorder"),
    ("drug addiction", "substance dependence"),

    # ===========================================
    # EYE DISEASES
    # ===========================================
    ("pink eye", "conjunctivitis"),
    ("eye infection", "conjunctivitis"),
    ("nearsightedness", "myopia"),
    ("farsightedness", "hyperopia"),
    ("old age vision", "presbyopia"),
    ("astigmatism", "refractive error"),
    ("lazy eye", "amblyopia"),
    ("crossed eyes", "strabismus"),
    ("cataracts", "lens opacity"),
    ("glaucoma", "increased eye pressure"),
    ("macular degeneration", "AMD"),
    ("diabetic eye disease", "diabetic retinopathy"),
    ("detached retina", "retinal detachment"),
    ("dry eyes", "dry eye syndrome"),
    ("eye inflammation", "uveitis"),
    ("color blindness", "color vision deficiency"),
    ("night blindness", "nyctalopia"),
    ("blindness", "vision loss"),
    ("double vision", "diplopia"),
    ("floaters", "vitreous floaters"),

    # ===========================================
    # EAR, NOSE, THROAT
    # ===========================================
    ("ear infection", "otitis media"),
    ("swimmer's ear", "otitis externa"),
    ("hearing loss", "hearing impairment"),
    ("deafness", "hearing loss"),
    ("ringing in ears", "tinnitus"),
    ("ear ringing", "tinnitus"),
    ("vertigo", "vestibular disorder"),
    ("motion sickness", "kinetosis"),
    ("nosebleed", "epistaxis"),
    ("runny nose", "rhinorrhea"),
    ("stuffy nose", "nasal congestion"),
    ("sinus infection", "sinusitis"),
    ("hay fever", "allergic rhinitis"),
    ("seasonal allergies", "allergic rhinitis"),
    ("deviated septum", "nasal septum deviation"),
    ("sleep apnea", "obstructive sleep apnea"),
    ("snoring", "sleep-disordered breathing"),
    ("laryngitis", "voice box inflammation"),
    ("tonsillitis", "tonsil infection"),
    ("strep throat", "streptococcal pharyngitis"),
    ("swollen glands", "lymphadenopathy"),

    # ===========================================
    # BLOOD DISORDERS
    # ===========================================
    ("anemia", "low blood count"),
    ("iron deficiency", "iron deficiency anemia"),
    ("sickle cell", "sickle cell disease"),
    ("sickle cell", "sickle cell anemia"),
    ("hemophilia", "bleeding disorder"),
    ("blood clotting disorder", "coagulopathy"),
    ("low platelets", "thrombocytopenia"),
    ("high platelets", "thrombocytosis"),
    ("leukemia", "blood cancer"),
    ("lymphoma", "lymph cancer"),
    ("sepsis", "blood poisoning"),
    ("sepsis", "septicemia"),
    ("blood infection", "bacteremia"),

    # ===========================================
    # IMMUNE SYSTEM
    # ===========================================
    ("allergy", "allergic reaction"),
    ("food allergy", "food hypersensitivity"),
    ("drug allergy", "drug hypersensitivity"),
    ("anaphylaxis", "severe allergic reaction"),
    ("autoimmune disease", "autoimmune disorder"),
    ("lupus", "systemic lupus erythematosus"),
    ("rheumatoid arthritis", "autoimmune arthritis"),
    ("multiple sclerosis", "autoimmune demyelination"),
    ("weak immune system", "immunodeficiency"),
    ("hiv", "human immunodeficiency virus"),
    ("aids", "acquired immunodeficiency syndrome"),

    # ===========================================
    # WOMEN'S HEALTH
    # ===========================================
    ("menstrual cramps", "dysmenorrhea"),
    ("painful periods", "dysmenorrhea"),
    ("heavy periods", "menorrhagia"),
    ("irregular periods", "oligomenorrhea"),
    ("no periods", "amenorrhea"),
    ("pms", "premenstrual syndrome"),
    ("menopause symptoms", "menopausal syndrome"),
    ("hot flashes", "vasomotor symptoms"),
    ("endometriosis", "uterine tissue disorder"),
    ("fibroids", "uterine fibroids"),
    ("fibroids", "leiomyoma"),
    ("ovarian cyst", "ovarian cystic disease"),
    ("pcos", "polycystic ovary syndrome"),
    ("pelvic inflammatory disease", "PID"),
    ("yeast infection", "vaginal candidiasis"),
    ("bacterial vaginosis", "BV"),
    ("ectopic pregnancy", "tubal pregnancy"),
    ("miscarriage", "spontaneous abortion"),
    ("preeclampsia", "pregnancy-induced hypertension"),
    ("gestational diabetes", "pregnancy diabetes"),

    # ===========================================
    # CHILDREN'S DISEASES
    # ===========================================
    ("chickenpox", "varicella"),
    ("measles", "rubeola"),
    ("mumps", "parotitis"),
    ("whooping cough", "pertussis"),
    ("croup", "laryngotracheobronchitis"),
    ("hand foot mouth disease", "HFMD"),
    ("roseola", "sixth disease"),
    ("fifth disease", "erythema infectiosum"),
    ("scarlet fever", "scarlatina"),
    ("ear infection", "otitis media"),
    ("pink eye", "conjunctivitis"),
    ("diaper rash", "diaper dermatitis"),
    ("cradle cap", "infantile seborrheic dermatitis"),
    ("colic", "infantile colic"),
    ("failure to thrive", "FTT"),
    ("jaundice in newborn", "neonatal jaundice"),
]

print(f"✅ Created {len(training_synonyms)} synonym pairs")

# Add all synonym terms to candidate bank
existing = set(c.lower() for c in candidate_bank)
added = 0
for common, medical in training_synonyms:
    if common.lower() not in existing:
        candidate_bank.append(common)
        existing.add(common.lower())
        added += 1
    if medical.lower() not in existing:
        candidate_bank.append(medical)
        existing.add(medical.lower())
        added += 1

print(f"✅ Added {added} terms to candidate bank")
print(f"   Total candidates: {len(candidate_bank)}")

✅ Created 423 synonym pairs
✅ Added 640 terms to candidate bank
   Total candidates: 2542


In [ ]:
#
#  Create Training Pairs (Dataset + Synonyms)
#

def create_contrastive_pairs(dataset_split):
    """Create (query, positive) pairs from dataset."""
    pairs = []

    for sample in dataset_split:
        tokens = sample['tokens']
        tags = sample['ner_tags']

        i = 0
        while i < len(tokens):
            if tags[i] == 1:  # B-Disease
                entity_start = i
                entity_tokens = [tokens[i]]
                i += 1
                while i < len(tokens) and tags[i] == 2:
                    entity_tokens.append(tokens[i])
                    i += 1

                entity_text = ' '.join(entity_tokens)
                marked_tokens = tokens.copy()
                marked_tokens[entity_start] = '[E]' + marked_tokens[entity_start]
                marked_tokens[i-1] = marked_tokens[i-1] + '[/E]'
                marked_sentence = ' '.join(marked_tokens)

                pairs.append({'query': marked_sentence, 'positive': entity_text})
            else:
                i += 1

    return pairs

# Create pairs from dataset
print("Creating contrastive pairs from dataset...")
train_pairs = create_contrastive_pairs(dataset['train'])
val_pairs = create_contrastive_pairs(dataset['validation'])
test_pairs = create_contrastive_pairs(dataset['test'])

print(f"   Dataset pairs: {len(train_pairs)}")

# Create pairs from synonyms
print("Creating synonym training pairs...")
synonym_pairs = []
for common, medical in training_synonyms:
    # Common → Medical
    synonym_pairs.append({'query': f"Patient diagnosed with [E]{common}[/E].", 'positive': medical})
    synonym_pairs.append({'query': f"Treatment for [E]{common}[/E] started.", 'positive': medical})
    # Medical → Common
    synonym_pairs.append({'query': f"Patient diagnosed with [E]{medical}[/E].", 'positive': common})
    synonym_pairs.append({'query': f"The [E]{medical}[/E] was confirmed.", 'positive': common})

print(f"   Synonym pairs: {len(synonym_pairs)}")

# Combine and shuffle (repeat synonyms 3x for emphasis)
combined_train_pairs = train_pairs + synonym_pairs * 3
random.shuffle(combined_train_pairs)

print(f"\n✅ Total training pairs: {len(combined_train_pairs)}")

Creating contrastive pairs from dataset...
   Dataset pairs: 5145
Creating synonym training pairs...
   Synonym pairs: 1692

✅ Total training pairs: 10221


In [ ]:
#
# Define ContrastiveDataset Class
#

class ContrastiveDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=128):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]

        query_enc = self.tokenizer(
            pair['query'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        pos_enc = self.tokenizer(
            pair['positive'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'query_input_ids': query_enc['input_ids'].squeeze(),
            'query_attention_mask': query_enc['attention_mask'].squeeze(),
            'pos_input_ids': pos_enc['input_ids'].squeeze(),
            'pos_attention_mask': pos_enc['attention_mask'].squeeze(),
        }

print("✅ ContrastiveDataset class defined!")

✅ ContrastiveDataset class defined!


In [ ]:
#
# CELL 4.5: Define InfoNCE Loss Function
#

def infonce_loss(query_emb, pos_emb, temperature=0.05):
    """
    InfoNCE contrastive loss with in-batch negatives.
    """
    # Normalize embeddings
    query_emb = F.normalize(query_emb, p=2, dim=-1)
    pos_emb = F.normalize(pos_emb, p=2, dim=-1)

    # Compute similarity matrix
    sim_matrix = torch.mm(query_emb, pos_emb.t()) / temperature

    # Labels: diagonal elements are positives
    labels = torch.arange(query_emb.size(0), device=query_emb.device)

    # Cross entropy loss
    loss = F.cross_entropy(sim_matrix, labels)

    return loss

print("✅ InfoNCE loss function defined!")

✅ InfoNCE loss function defined!


In [ ]:
#
#  Load and Initialize Contrastive Model
#

print(f"🔄 Loading SapBERT model: {Config.CONTRASTIVE_MODEL}")

# Load tokenizer
contrastive_tokenizer = AutoTokenizer.from_pretrained(Config.CONTRASTIVE_MODEL)

# Add special tokens for entity markers
special_tokens = {'additional_special_tokens': ['[E]', '[/E]']}
contrastive_tokenizer.add_special_tokens(special_tokens)

# Load model
contrastive_model = AutoModel.from_pretrained(Config.CONTRASTIVE_MODEL)
contrastive_model.resize_token_embeddings(len(contrastive_tokenizer))
contrastive_model.to(device)

print(f"✅ Model loaded on {device}")
print(f"   Parameters: {sum(p.numel() for p in contrastive_model.parameters()):,}")

🔄 Loading SapBERT model: cambridgeltl/SapBERT-from-PubMedBERT-fulltext


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


✅ Model loaded on cuda
   Parameters: 109,483,776


In [ ]:
#
#  Train Contrastive Model
#

print(" Starting Stage 2 Training (Contrastive Learning)...")
print(f"   Training pairs: {len(combined_train_pairs)}")
print(f"   Epochs: {Config.EPOCHS_CONTRASTIVE}")
print(f"   Temperature: {Config.TEMPERATURE}")

# Create datasets
train_dataset = ContrastiveDataset(combined_train_pairs, contrastive_tokenizer)
val_dataset = ContrastiveDataset(val_pairs, contrastive_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE_CONTRASTIVE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE_CONTRASTIVE)

# Optimizer
optimizer = torch.optim.AdamW(contrastive_model.parameters(), lr=Config.LR_CONTRASTIVE, weight_decay=0.01)

# Training history
history = {'train_loss': [], 'val_loss': []}

# Training loop
for epoch in range(Config.EPOCHS_CONTRASTIVE):
    contrastive_model.train()
    total_train_loss = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS_CONTRASTIVE}")
    for batch in progress:
        optimizer.zero_grad()

        # Get embeddings
        query_out = contrastive_model(
            input_ids=batch['query_input_ids'].to(device),
            attention_mask=batch['query_attention_mask'].to(device)
        )
        query_emb = query_out.last_hidden_state[:, 0, :]

        pos_out = contrastive_model(
            input_ids=batch['pos_input_ids'].to(device),
            attention_mask=batch['pos_attention_mask'].to(device)
        )
        pos_emb = pos_out.last_hidden_state[:, 0, :]

        # Compute loss
        loss = infonce_loss(query_emb, pos_emb, Config.TEMPERATURE)

        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(contrastive_model.parameters(), 1.0)
        optimizer.step()

        total_train_loss += loss.item()
        progress.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = total_train_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)

    # Validation
    contrastive_model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            query_out = contrastive_model(
                input_ids=batch['query_input_ids'].to(device),
                attention_mask=batch['query_attention_mask'].to(device)
            )
            query_emb = query_out.last_hidden_state[:, 0, :]

            pos_out = contrastive_model(
                input_ids=batch['pos_input_ids'].to(device),
                attention_mask=batch['pos_attention_mask'].to(device)
            )
            pos_emb = pos_out.last_hidden_state[:, 0, :]

            loss = infonce_loss(query_emb, pos_emb, Config.TEMPERATURE)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    history['val_loss'].append(avg_val_loss)

    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

print("\n✅ Stage 2 Training Complete!")

 Starting Stage 2 Training (Contrastive Learning)...


NameError: name 'combined_train_pairs' is not defined

In [ ]:
#
#  Build FAISS Index
#

print("🔧 Building FAISS index...")

contrastive_model.eval()
candidate_embeddings = []

for i in tqdm(range(0, len(candidate_bank), 64), desc="Encoding candidates"):
    batch = candidate_bank[i:i+64]
    enc = contrastive_tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')

    with torch.no_grad():
        emb = contrastive_model(
            enc['input_ids'].to(device),
            enc['attention_mask'].to(device)
        ).last_hidden_state[:, 0, :]
        candidate_embeddings.append(emb.cpu().numpy())

candidate_embeddings = np.vstack(candidate_embeddings)
candidate_embeddings = candidate_embeddings / np.linalg.norm(candidate_embeddings, axis=1, keepdims=True)
candidate_embeddings = candidate_embeddings.astype(np.float32)

# Build index
faiss_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
faiss_index.add(candidate_embeddings)

print(f"\n✅ FAISS index built with {faiss_index.ntotal} candidates")

🔧 Building FAISS index...


Encoding candidates:   0%|          | 0/40 [00:00<?, ?it/s]


✅ FAISS index built with 2542 candidates


In [ ]:
#
# Define Retrieval Function
#

def retrieve_candidates(query_text, top_k=5):
    """Retrieve top-k candidates for a query."""
    contrastive_model.eval()

    encoded = contrastive_tokenizer(
        query_text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

    with torch.no_grad():
        outputs = contrastive_model(
            input_ids=encoded['input_ids'].to(device),
            attention_mask=encoded['attention_mask'].to(device)
        )
        query_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()

    query_emb = query_emb / np.linalg.norm(query_emb)
    query_emb = query_emb.astype(np.float32)

    scores, indices = faiss_index.search(query_emb, top_k)

    results = []
    for idx, score in zip(indices[0], scores[0]):
        results.append({'candidate': candidate_bank[idx], 'score': float(score)})

    return results

# Test retrieval
print("🔬 Testing retrieval...")
test_query = "Patient diagnosed with [E]heart attack[/E]."
results = retrieve_candidates(test_query)

print(f"\nQuery: {test_query}")
print("Top-5 Results:")
for i, r in enumerate(results, 1):
    print(f"   {i}. {r['candidate']} (score: {r['score']:.3f})")

🔬 Testing retrieval...

Query: Patient diagnosed with [E]heart attack[/E].
Top-5 Results:
   1. heart attack (score: 0.957)
   2. cardiac infarction (score: 0.938)
   3. myocardial infarction (score: 0.923)
   4. MI (score: 0.921)
   5. ischemic heart disease (score: 0.837)



# SECTION 5: EVALUATION


In [ ]:
#
# Evaluate Retrieval on Test Set
#

print("📊 Evaluating Stage 2 Retrieval on Test Set...")

def evaluate_retrieval(pairs, top_k=10):
    mrrs, recall_1, recall_5, recall_10 = [], [], [], []

    for pair in tqdm(pairs, desc="Evaluating"):
        target = pair['positive'].lower()
        results = retrieve_candidates(pair['query'], top_k=top_k)

        rank = None
        for i, r in enumerate(results, 1):
            if r['candidate'].lower() == target:
                rank = i
                break

        if rank:
            mrrs.append(1.0 / rank)
            recall_1.append(1 if rank <= 1 else 0)
            recall_5.append(1 if rank <= 5 else 0)
            recall_10.append(1 if rank <= 10 else 0)
        else:
            mrrs.append(0)
            recall_1.append(0)
            recall_5.append(0)
            recall_10.append(0)

    return {
        'MRR': np.mean(mrrs),
        'Recall@1': np.mean(recall_1),
        'Recall@5': np.mean(recall_5),
        'Recall@10': np.mean(recall_10)
    }

stage2_results = evaluate_retrieval(test_pairs)

print("\n" + "=" * 50)
print(" STAGE 2 RETRIEVAL RESULTS")
print("=" * 50)
for metric, value in stage2_results.items():
    print(f"   {metric}: {value:.4f}")
print("=" * 50)

📊 Evaluating Stage 2 Retrieval on Test Set...


Evaluating:   0%|          | 0/960 [00:00<?, ?it/s]


 STAGE 2 RETRIEVAL RESULTS
   MRR: 0.6457
   Recall@1: 0.6385
   Recall@5: 0.6542
   Recall@10: 0.6542


In [ ]:
#
# Test on Hard Cases (Synonyms)
#

print("📊 Testing on Hard Cases (Synonym Pairs)...")

hard_test_cases = [
    ("heart attack", "myocardial infarction"),
    ("high blood pressure", "hypertension"),
    ("stroke", "cerebrovascular accident"),
    ("flu", "influenza"),
    ("diabetes", "diabetes mellitus"),
    ("kidney failure", "renal failure"),
    ("chickenpox", "varicella"),
    ("pink eye", "conjunctivitis"),
]

print(f"\n{'Common Name':<25} {'Medical Name':<30} {'Found?':<10}")
print("-" * 65)

successes = 0
for common, medical in hard_test_cases:
    query = f"Patient has [E]{common}[/E]."
    results = retrieve_candidates(query, top_k=5)

    top_5 = [r['candidate'].lower() for r in results]
    found = medical.lower() in top_5

    if found:
        successes += 1
        status = "✅ Yes"
    else:
        status = "❌ No"

    print(f"{common:<25} {medical:<30} {status}")

print("-" * 65)
print(f"\n Success Rate: {successes}/{len(hard_test_cases)} ({100*successes/len(hard_test_cases):.0f}%)")

📊 Testing on Hard Cases (Synonym Pairs)...

Common Name               Medical Name                   Found?    
-----------------------------------------------------------------
heart attack              myocardial infarction          ✅ Yes
high blood pressure       hypertension                   ✅ Yes
stroke                    cerebrovascular accident       ✅ Yes
flu                       influenza                      ✅ Yes
diabetes                  diabetes mellitus              ✅ Yes
kidney failure            renal failure                  ✅ Yes
chickenpox                varicella                      ✅ Yes
pink eye                  conjunctivitis                 ✅ Yes
-----------------------------------------------------------------

 Success Rate: 8/8 (100%)


#
# SECTION 6: SAVE MODEL
#

In [ ]:
#
# Save Trained Model
#

save_dir = f"{Config.PROJECT_DIR}/models/stage2_final"
os.makedirs(save_dir, exist_ok=True)

# Save model weights
torch.save(contrastive_model.state_dict(), f"{save_dir}/model.pt")
print("✅ Model weights saved")

# Save tokenizer
contrastive_tokenizer.save_pretrained(f"{save_dir}/tokenizer")
print("✅ Tokenizer saved")

# Save candidate bank
with open(f"{save_dir}/candidate_bank.json", 'w') as f:
    json.dump(candidate_bank, f)
print(f"✅ Candidate bank saved ({len(candidate_bank)} candidates)")

# Save FAISS index
faiss.write_index(faiss_index, f"{save_dir}/faiss_index.bin")
print("✅ FAISS index saved")

# Save embeddings
np.save(f"{save_dir}/candidate_embeddings.npy", candidate_embeddings)
print("✅ Embeddings saved")

# Save config
config_to_save = {
    'model_name': Config.CONTRASTIVE_MODEL,
    'temperature': Config.TEMPERATURE,
    'epochs': Config.EPOCHS_CONTRASTIVE,
    'num_candidates': len(candidate_bank),
    'embedding_dim': int(candidate_embeddings.shape[1])
}
with open(f"{save_dir}/config.json", 'w') as f:
    json.dump(config_to_save, f, indent=2)
print("✅ Config saved")

print(f"\n🎉 All files saved to: {save_dir}")

✅ Model weights saved
✅ Tokenizer saved
✅ Candidate bank saved (2542 candidates)
✅ FAISS index saved
✅ Embeddings saved
✅ Config saved

🎉 All files saved to: /content/drive/MyDrive/medical_ner/models/stage2_final


In [ ]:
#
# Full Pipeline Demo
#

def demo_full_pipeline(text):
    """Run complete two-stage pipeline."""
    print(f"\n{'='*60}")
    print(f"📝 Input: {text}")
    print(f"{'='*60}")

    # Stage 1: NER
    print("\n🔬 Stage 1: NER")
    entities = ner_pipeline(text)

    if not entities:
        print("   No entities found.")
        return

    for ent in entities:
        print(f"   Found: '{ent['word']}' [{ent['entity_group']}]")

    # Stage 2: Normalization
    print("\n🔗 Stage 2: Normalization")
    for ent in entities:
        query = text.replace(ent['word'], f"[E]{ent['word']}[/E]", 1)
        results = retrieve_candidates(query, top_k=3)

        print(f"   '{ent['word']}' → '{results[0]['candidate']}' (score: {results[0]['score']:.3f})")

# Test
demo_full_pipeline("The patient has diabetes and high blood pressure.")
demo_full_pipeline("She was diagnosed with breast cancer last year.")
demo_full_pipeline("Patient presents with flu symptoms and chest pain.")


📝 Input: The patient has diabetes and high blood pressure.

🔬 Stage 1: NER
   Found: 'diabetes' [Disease]

🔗 Stage 2: Normalization
   'diabetes' → 'diabetes mellitus' (score: 0.862)

📝 Input: She was diagnosed with breast cancer last year.

🔬 Stage 1: NER
   Found: 'breast cancer' [Disease]

🔗 Stage 2: Normalization
   'breast cancer' → 'breast carcinoma' (score: 0.928)

📝 Input: Patient presents with flu symptoms and chest pain.

🔬 Stage 1: NER
   Found: 'flu symptoms' [Disease]
   Found: 'chest pain' [Disease]

🔗 Stage 2: Normalization
   'flu symptoms' → 'influenza' (score: 0.771)
   'chest pain' → 'chest pain' (score: 0.926)


In [ ]:
#
# Final Summary
#

print("\n" + "=" * 70)
print(" FINAL SUMMARY")
print("=" * 70)

print(f"\n STAGE 1 (NER):")
print(f"   F1: {stage1_results['f1']:.4f}")
print(f"   Precision: {stage1_results['precision']:.4f}")
print(f"   Recall: {stage1_results['recall']:.4f}")

print(f"\n🔗 STAGE 2 (Retrieval):")
for metric, value in stage2_results.items():
    print(f"   {metric}: {value:.4f}")

print(f"\n Models saved to: {Config.PROJECT_DIR}/models/")
print("=" * 70)
print(" PROJECT COMPLETE!")
print("=" * 70)


 FINAL SUMMARY

 STAGE 1 (NER):
   F1: 0.8647
   Precision: 0.8330
   Recall: 0.8990

🔗 STAGE 2 (Retrieval):
   MRR: 0.6457
   Recall@1: 0.6385
   Recall@5: 0.6542
   Recall@10: 0.6542

 Models saved to: /content/drive/MyDrive/medical_ner/models/
 PROJECT COMPLETE!
